## Transformations

A number of transformations are available for Futures. We can look at some of those.

In [1]:
import scala.util.Random
import scala.concurrent.Future
import concurrent.ExecutionContext.Implicits.global

def giveMeANumber = Future{
    Thread.sleep(4000)
    Random.nextInt(5) 
}

def giveMeABadNumber = Future{
    Thread.sleep(4000)
    throw Exception("bad")
    Random.nextInt(5) 
}

import scala.util.Random

import scala.concurrent.Future

import concurrent.ExecutionContext.Implicits.global


defined function giveMeANumber
defined function giveMeABadNumber


### failed

It tries to transform a failed Future into a successfully completed one with Throwable as a result. 

In [3]:
giveMeANumber.failed

res2: Future[Throwable] = Failure(
  exception = scala.concurrent.Future$$anon$3: Future.failed not completed with a throwable.
)

If Future completes successfully, then the resulting Future will fail with an Exception.

In [ ]:
giveMeANumber.failed

### fallbackTo

`fallbackTo` takes an alternative Future in case of a failure of the current one and evaluates them simultaneously. If both fail, the resulting Future will fail with the Throwable taken from the current one.

In [5]:
giveMeABadNumber.fallbackTo(
    giveMeABadNumber
)

res4: Future[Int] = Failure(exception = java.lang.Exception: bad)

In [6]:
giveMeANumber.fallbackTo(
    Future.successful(20)
)

res5: Future[Int] = Success(value = 4)

In [7]:
giveMeABadNumber.fallbackTo(
    Future.failed(Exception("nothing works"))
)

res6: Future[Int] = Failure(exception = java.lang.Exception: bad)

In [8]:
Future{
    println("original future")
    10
}.fallbackTo{
    println("alternative future")
    Future.failed(Exception("nothing works"))
}

alternative future
original future


res7: Future[Int] = Success(value = 10)

### recover

`recover` takes a partial function that turns any matching exception into a successful result. Otherwise, it will keep the original exception.



In [9]:
val f:Future[Int] = Future{
    println("original future")
    throw Exception("nothing works")
    10
}

f.recover{
    case ex => 
        println("there was a problem")
        20
}

original future
there was a problem


f: Future[Int] = Failure(exception = java.lang.Exception: nothing works)
res8_1: Future[Int] = [running]

### map

It transforms the successful result of a Future without blocking the main thread:


In [11]:
val f = Future(30)
f.map(i=>i+10)

f: Future[Int] = Success(value = 30)
res10_1: Future[Int] = Success(value = 40)

In [13]:
Future{
    Thread.sleep(3000)
    32+4 
} map (_*2)

res12: Future[Int] = Success(value = 72)

### flatmap

It behaves in the same way as the map method but keeps the resulting Future flat


In [14]:

val f = Future(30)
f.flatMap(i=>Future(i+10))

f: Future[Int] = Success(value = 30)
res13_1: Future[Int] = Success(value = 40)

In [15]:
def f:Future[Int] = Future{
    throw Exception("bad things")
    10
}

f.map(i => i+10)

defined function f
res14_1: Future[Int] = Failure(exception = java.lang.Exception: bad things)

In [16]:
f.flatMap(i => Future(i+10))

res15: Future[Int] = Failure(exception = java.lang.Exception: bad things)

In [17]:
def g:Future[Int] = Future(20)

defined function g

In [18]:
val list = List(g,f)

list: List[Future[Int]] = List(
  Success(value = 20),
  Failure(exception = java.lang.Exception: bad things)
)

In [19]:
list.map(ff => ff.map(i=>i+10))

res18: List[Future[Int]] = List(
  Success(value = 30),
  Failure(exception = java.lang.Exception: bad things)
)

### transform

The `transform()` method accepts a function that takes the `Future` result as the input and returns a `Try` instance as the output. 

In [20]:
import scala.util.{Success,Failure}

val f = Future.successful(15)

import scala.util.{Success,Failure}


f: Future[Int] = Success(value = 15)

In [21]:
f.transform {
    case Success(v) => Success(v+1)
    case Failure(ex) => Failure(new IllegalArgumentException)
}

res20: Future[Int] = Success(value = 16)

### zip

It tries to combine the successful results of both Futures into a pair. If any of them fail, the resulting `Future` will also fail with the same reason as the leftmost of them.

In [22]:
giveMeANumber.zip(giveMeABadNumber)

res21: Future[(Int, Int)] = Failure(exception = java.lang.Exception: bad)

In [23]:
giveMeANumber.zip(giveMeANumber)

res22: Future[(Int, Int)] = Success(value = (4, 2))

### traverse

`traverse` performs a parallel map of multiple elements. If any of them fail, the resulting `Future` will also fail.

In [25]:
val list = List(3,6,7,2)

list: List[Int] = List(3, 6, 7, 2)

In [26]:
def complexOperation(i:Int) = Future{
    Thread.sleep(2000)
    i*i 
}

defined function complexOperation

In [28]:
Future.traverse(list)(complexOperation)

res27: Future[List[Int]] = Success(value = List(9, 36, 49, 4))

In [29]:
def complexOperation(i:Int) = Future{
    if i==6 then throw Exception("6 not accepted")
    Thread.sleep(2000)
    i*i 
}

defined function complexOperation

In [30]:
Future.traverse(list)(complexOperation)

res29: Future[List[Int]] = Failure(exception = java.lang.Exception: 6 not accepted)

### for yield

To run multiple computations in parallel and join their results when all of the futures have been completed, we can use a `for` expression.

In [32]:
val f1 = complexOperation(1)
val f2 = complexOperation(3)
val f3 = complexOperation(4)

val result = 
    for
        r1 <- f1
        r2 <- f2
        r3 <- f3
    yield
        (r1 + r2 + r3)
        

f1: Future[Int] = Success(value = 1)
f2: Future[Int] = Success(value = 9)
f3: Future[Int] = Success(value = 16)
result: Future[Int] = Success(value = 26)

In [33]:
val result = 
    for
        r1 <- complexOperation(1)
        r2 <- complexOperation(3)
        r3 <- complexOperation(4)
    yield
        (r1 + r2 + r3)

result: Future[Int] = Success(value = 26)